In [ ]:
import json
import requests

def download_ids(name):
    url = f"https://raw.githubusercontent.com/purarue/mal-id-cache/master/cache/{name}_cache.json"
    ids = []
    data = requests.get(url).json()
    if isinstance(data, dict):
        for arr in data.values():
            ids.extend(arr)
    else:
        ids = data
    return ids

mal_anime_ids = download_ids("anime")
mal_manga_ids = download_ids("manga")
mal_char_ids = download_ids("character")
mal_people_ids = download_ids("people")


In [ ]:
import json
import requests
import time
import os

CACHE_URL = "https://raw.githubusercontent.com/purarue/mal-id-cache/master/cache/anime_cache.json"
JIKAN_ANIME_FULL_URL = "https://api.jikan.moe/v4/anime/{}/full"

OUTPUT_JSON = "all_anime_full.json"
CHECKPOINT_JSON = "progress_checkpoint.json"
SKIPPED_TXT = "skipped_anime_ids.txt"
BATCH_SIZE = 10
REQUEST_DELAY = 0.35

def get_all_mal_ids():
    cache = requests.get(CACHE_URL).json()
    return cache["sfw"] + cache["nsfw"]

def load_checkpoint():
    animelist, processed_ids, skipped = [], set(), []
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, encoding="utf-8") as f:
            animelist = json.load(f)
            processed_ids = {entry['mal_id'] for entry in animelist if 'mal_id' in entry}
    if os.path.exists(CHECKPOINT_JSON):
        with open(CHECKPOINT_JSON) as f:
            check = json.load(f)
            processed_ids.update(check.get("processed_ids", []))
            skipped.extend(check.get("skipped", []))
    return animelist, processed_ids, skipped

def save_checkpoint(animelist, processed_ids, skipped):
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(animelist, f, ensure_ascii=False, indent=2)
    with open(CHECKPOINT_JSON, "w") as f:
        json.dump({
            "processed_ids": list(processed_ids),
            "skipped": skipped
        }, f, indent=2)
    if skipped:
        with open(SKIPPED_TXT, "w") as sf:
            sf.write("\n".join(str(x) for x in skipped))

def fetch_anime_full(mal_id):
    url = JIKAN_ANIME_FULL_URL.format(mal_id)
    try:
        resp = requests.get(url, timeout=20)
        if resp.status_code == 200:
            return resp.json()
        else:
            print(f"Error for {mal_id}: HTTP {resp.status_code}")
            return None
    except Exception as e:
        print(f"Failed on {mal_id}: {e}")
        return None

def main():
    mal_ids = get_all_mal_ids()
    animelist, processed_ids, skipped = load_checkpoint()
    print(f"Found {len(animelist)} already processed. Checkpointing enabled.")

    to_process = [mid for mid in mal_ids if mid not in processed_ids and mid not in skipped]

    for i, mal_id in enumerate(to_process, 1):
        print(f"({i}/{len(to_process)}) Fetching MAL ID {mal_id}")
        anime_data = fetch_anime_full(mal_id)
        if anime_data and "data" in anime_data:
            animelist.append(anime_data["data"])
            processed_ids.add(mal_id)
        else:
            skipped.append(mal_id)
        if i % BATCH_SIZE == 0:
            save_checkpoint(animelist, processed_ids, skipped)
        time.sleep(REQUEST_DELAY)

    # Final save
    save_checkpoint(animelist, processed_ids, skipped)
    print(f"Done. Saved {len(animelist)} items to {OUTPUT_JSON}.")

if __name__ == "__main__":
    main()


In [7]:
def main():
    animelist, processed_ids, _ = load_checkpoint()
    print(f"Found {len(animelist)} already processed. Retrying skipped IDs.")

    # Load skipped IDs from file
    if not os.path.exists(SKIPPED_TXT):
        print("No skipped_anime_ids.txt found.")
        return

    with open(SKIPPED_TXT) as f:
        skipped_ids = [int(line.strip()) for line in f if line.strip().isdigit()]

    to_process = [mid for mid in skipped_ids if mid not in processed_ids]
    print(f"{len(to_process)} skipped IDs to retry.")

    skipped = []  # re-init skipped list to track only those that fail again

    for i, mal_id in enumerate(to_process, 1):
        print(f"({i}/{len(to_process)}) Retrying MAL ID {mal_id}")
        anime_data = fetch_anime_full(mal_id)
        if anime_data and "data" in anime_data:
            animelist.append(anime_data["data"])
            processed_ids.add(mal_id)
        else:
            skipped.append(mal_id)
        if i % BATCH_SIZE == 0:
            save_checkpoint(animelist, processed_ids, skipped)
        time.sleep(REQUEST_DELAY)

    # Final save
    save_checkpoint(animelist, processed_ids, skipped)
    print(f"Retry complete. Total saved: {len(animelist)}. Still skipped: {len(skipped)}.")
main()

Found 661 already processed. Retrying skipped IDs.
546 skipped IDs to retry.
(1/546) Retrying MAL ID 51410
(2/546) Retrying MAL ID 55098
(3/546) Retrying MAL ID 55099
(4/546) Retrying MAL ID 55400
(5/546) Retrying MAL ID 55459
(6/546) Retrying MAL ID 55527
(7/546) Retrying MAL ID 55610
(8/546) Retrying MAL ID 56013
(9/546) Retrying MAL ID 56198
(10/546) Retrying MAL ID 56509
(11/546) Retrying MAL ID 56636
(12/546) Retrying MAL ID 56637
(13/546) Retrying MAL ID 56815
(14/546) Retrying MAL ID 56867
(15/546) Retrying MAL ID 57072
(16/546) Retrying MAL ID 57155
(17/546) Retrying MAL ID 57421
(18/546) Retrying MAL ID 57536
(19/546) Retrying MAL ID 58077
(20/546) Retrying MAL ID 58181
(21/546) Retrying MAL ID 58373
(22/546) Retrying MAL ID 58438
(23/546) Retrying MAL ID 58440
(24/546) Retrying MAL ID 58441
(25/546) Retrying MAL ID 58474
(26/546) Retrying MAL ID 58475
(27/546) Retrying MAL ID 58477
(28/546) Retrying MAL ID 58478
(29/546) Retrying MAL ID 58481
(30/546) Retrying MAL ID 58539
(3

In [4]:
def main():
    # Step 1: Load latest MAL IDs from GitHub
    mal_ids = sorted(get_all_mal_ids())  # Always fetch fresh

    # Step 2: Load previously saved data
    animelist, processed_ids, skipped = load_checkpoint()
    print(f"Already processed: {len(animelist)} entries.")

    # Step 3: Find last processed ID (or start from beginning)
    last_id = max(processed_ids) if processed_ids else 0
    print(f"Resuming from MAL ID > {last_id}")

    # Step 4: Process only new IDs
    to_process = [mid for mid in mal_ids if mid > last_id and mid not in skipped]
    print(f"{len(to_process)} new IDs to process...")

    for i, mal_id in enumerate(to_process, 1):
        print(f"({i}/{len(to_process)}) Fetching MAL ID {mal_id}")
        anime_data = fetch_anime_full(mal_id)
        if anime_data and "data" in anime_data:
            animelist.append(anime_data["data"])
            processed_ids.add(mal_id)
        else:
            skipped.append(mal_id)

        if i % BATCH_SIZE == 0:
            save_checkpoint(animelist, processed_ids, skipped)

        time.sleep(REQUEST_DELAY)

    # Final save
    save_checkpoint(animelist, processed_ids, skipped)
    print(f"Finished. Total anime saved: {len(animelist)}. Skipped: {len(skipped)}.")
main()

Already processed: 901 entries.
Resuming from MAL ID > 9999


TypeError: '>' not supported between instances of 'int' and 'str'

In [1]:
#Characters seeding

In [3]:
import json
import requests
import time
import os

CACHE_URL = "https://raw.githubusercontent.com/purarue/mal-id-cache/master/cache/anime_cache.json"
JIKAN_CHARACTERS_URL = "https://api.jikan.moe/v4/anime/{}/characters"

OUTPUT_JSON = "anime_characters.json"
CHECKPOINT_JSON = "progress_characters_checkpoint.json"
SKIPPED_TXT = "skipped_character_ids.txt"
BATCH_SIZE = 20
REQUEST_DELAY = 0.4      # Per Jikan docs, recommended ~0.33-0.5s

def get_all_mal_ids():
    cache = requests.get(CACHE_URL).json()
    return cache["sfw"] + cache["nsfw"]

def load_checkpoint():
    chardict, processed_ids, skipped = {}, set(), []
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, encoding="utf-8") as f:
            chardict = json.load(f)
            processed_ids = set(str(mid) for mid in chardict)  # keys are str
    if os.path.exists(CHECKPOINT_JSON):
        with open(CHECKPOINT_JSON) as f:
            check = json.load(f)
            processed_ids.update(str(x) for x in check.get("processed_ids", []))
            skipped.extend(str(x) for x in check.get("skipped", []))
    return chardict, processed_ids, skipped

def save_checkpoint(chardict, processed_ids, skipped):
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(chardict, f, ensure_ascii=False, indent=1)
    with open(CHECKPOINT_JSON, "w") as f:
        json.dump({
            "processed_ids": list(processed_ids),
            "skipped": skipped
        }, f, indent=2)
    if skipped:
        with open(SKIPPED_TXT, "w") as sf:
            sf.write("\n".join(str(x) for x in skipped))

def fetch_characters(mal_id):
    url = JIKAN_CHARACTERS_URL.format(mal_id)
    try:
        resp = requests.get(url, timeout=15)
        if resp.status_code == 200:
            return resp.json().get("data", [])
        elif resp.status_code in (404, 400, 204):
            # 404: Not found, 204: no characters, 400: invalid
            print(f"No characters found for {mal_id}.")
            return []
        else:
            print(f"Error for {mal_id}: HTTP {resp.status_code}")
            return None
    except Exception as e:
        print(f"Failed on {mal_id}: {e}")
        return None

def main():
    mal_ids = get_all_mal_ids()
    chardict, processed_ids, skipped = load_checkpoint()
    print(f"Characters: {len(chardict)} already processed. Checkpointing enabled.")

    to_process = [mid for mid in mal_ids if str(mid) not in processed_ids and str(mid) not in skipped]
    total = len(to_process)
    failed_this_run = []

    for idx, mal_id in enumerate(to_process, 1):
        print(f"({idx}/{total}) Fetching characters for MAL ID {mal_id}")
        char_data = fetch_characters(mal_id)
        if char_data is not None:
            chardict[str(mal_id)] = char_data
            processed_ids.add(str(mal_id))
        else:
            skipped.append(str(mal_id))
            failed_this_run.append(mal_id)
        if idx % BATCH_SIZE == 0:
            save_checkpoint(chardict, processed_ids, skipped)
            print(f"=> Batch checkpointed. {len(processed_ids)} done, {len(skipped)} skipped so far.")
        time.sleep(REQUEST_DELAY)

    # Final save
    save_checkpoint(chardict, processed_ids, skipped)
    print(f"Done. Saved {len(chardict)} entries to {OUTPUT_JSON}.")
    if failed_this_run:
        print(f"[FAILED THIS RUN]: {failed_this_run}")

if __name__ == "__main__":
    main()


Characters: 901 already processed. Checkpointing enabled.
Done. Saved 901 entries to anime_characters.json.
